## pruebas

In [ ]:
import os
import webbrowser
import pandas as pd
import json
import geopandas as gpd
import colorsys
import numpy as np

# Cargar datos
usuario = os.getlogin()
ruta_clues = fr"C:\Users\{usuario}\IMSS-BIENESTAR\División de Procesamiento de información - Repositorio de Datos\CLUES\clues.parquet"
clues = pd.read_parquet(ruta_clues)
clues = clues.loc[
    (clues["nivel_atencion"].isin(["SEGUNDO NIVEL", "TERCER NIVEL"]))
]

# Cargar archivos
ruta_parquet = fr"C:\Users\{usuario}\Downloads\version_8_long.gpkg"
parquet = gpd.read_file(ruta_parquet)

ruta_nueva = fr"C:\Users\{usuario}\Downloads\cluster_17_rutas_geo.gpkg"
carretera = gpd.read_file(ruta_nueva)

In [2]:
carretera["nombre_cluster"] = carretera["nombre_cluster"].str.split("_").str[1]


In [ ]:
carretera = carretera.drop(columns="cluster_id", errors="ignore")

In [4]:
carretera=carretera.rename(columns={"nombre_cluster":"cluster_id"})

In [5]:

# carretera["cluster_id"] = (
#     carretera["cluster_id"]
#     .str.strip()
#     .replace("0", pd.NA)
# )


In [6]:
carretera["cluster_id"].dtype


dtype('O')

In [7]:
parquet = parquet.rename(
    columns={
        "clues_code": "clues_imb",
        "tipo_cluster": "type",
        
    }
)


In [8]:
carretera= carretera.rename(columns={"clues_imb_ancla":"clues_imb_hg"})

In [9]:
# Unir datos
parquet = parquet.merge(clues[['clues_imb', 'entidad', 'nombre_de_la_unidad']], on='clues_imb', how='left')

carretera_long = carretera.melt(id_vars=['cluster_id'], value_vars=['clues_imb_hbc', 'clues_imb_hg'], 
                               value_name='clues_imb').drop(columns='variable').dropna()

parquet = parquet.merge(carretera_long[['clues_imb', "cluster_id"]], on='clues_imb', how='left')

# Crear relaciones
relaciones = []
for idx, row in carretera.iterrows():
    cluster_id = row['cluster_id']
    ancla = row['clues_imb_hg']
    hbc = row['clues_imb_hbc']
    
    if pd.notna(ancla) and pd.notna(hbc) and pd.notna(cluster_id):
        relacion = {'cluster_id': cluster_id, 'clues_ancla': ancla, 'clues_imb_hbc': hbc}
        relaciones.append(relacion)

relaciones_df = pd.DataFrame(relaciones)

# Unir datos completos
relaciones_df = relaciones_df.merge(
    parquet[['clues_imb', 'latitud', 'longitud', 'entidad', 'nombre_de_la_unidad']],
    left_on='clues_ancla', right_on='clues_imb', how='left'
).rename(columns={
    'latitud': 'latitud_ancla', 'longitud': 'longitud_ancla', 
    'entidad': 'entidad_ancla', 'nombre_de_la_unidad': 'nombre_hospital_ancla'
}).drop(columns='clues_imb')

relaciones_df = relaciones_df.merge(
    parquet[['clues_imb', 'latitud', 'longitud', 'entidad', 'nombre_de_la_unidad']],
    left_on='clues_imb_hbc', right_on='clues_imb', how='left'
).rename(columns={
    'latitud': 'latitud_hbc', 'longitud': 'longitud_hbc',
    'entidad': 'entidad_hbc', 'nombre_de_la_unidad': 'nombre_hospital_hbc'
}).drop(columns='clues_imb')

# Obtener todos los estados únicos
relaciones_df_original = relaciones_df.copy()
estados = relaciones_df_original["entidad_ancla"].unique()

# Loop para generar HTML para cada estado
for estado in estados:
    relaciones_df = relaciones_df_original[relaciones_df_original["entidad_ancla"] == estado]

In [10]:
relaciones_df = relaciones_df_original.copy()

In [11]:
from pathlib import Path
import colorsys
import json

cluster_values = sorted({
    str(value).strip()
    for value in relaciones_df["cluster_id"].dropna()
    if str(value).strip()
})
cluster_color_map = {
    cluster_id: "#{:02x}{:02x}{:02x}".format(
        *(int(channel * 255) for channel in colorsys.hsv_to_rgb(index / max(len(cluster_values), 1), 0.7, 0.85))
    )
    for index, cluster_id in enumerate(cluster_values)
}

features = []
seen_points = set()
point_definitions = [
    ("ancla", "clues_ancla", "nombre_hospital_ancla", "latitud_ancla", "longitud_ancla", "entidad_ancla"),
    ("hbc", "clues_imb_hbc", "nombre_hospital_hbc", "latitud_hbc", "longitud_hbc", "entidad_hbc"),
]

for _, row in relaciones_df.iterrows():
    cluster_id = str(row.get("cluster_id", "")).strip()
    if not cluster_id or cluster_id.lower() == "nan":
        continue

    for point_type, clues_key, name_key, lat_key, lon_key, entity_key in point_definitions:
        clues_value = row.get(clues_key)
        lat_value = row.get(lat_key)
        lon_value = row.get(lon_key)
        if pd.isna(clues_value) or pd.isna(lat_value) or pd.isna(lon_value):
            continue

        clues = str(clues_value).strip()
        if not clues:
            continue
        point_key = (point_type, clues, cluster_id)
        if point_key in seen_points:
            continue
        seen_points.add(point_key)

        name = row.get(name_key)
        entity = row.get(entity_key)
        associated_anchor = row.get("clues_ancla") if point_type == "hbc" else ""
        features.append({
            "type": "Feature",
            "geometry": {
                "type": "Point",
                "coordinates": [float(lon_value), float(lat_value)],
            },
            "properties": {
                "id": f"{point_type}-{clues}-{cluster_id}",
                "tipo": point_type,
                "clues": clues,
                "nombre": "" if pd.isna(name) else str(name).strip(),
                "entidad": "" if pd.isna(entity) else str(entity).strip(),
                "cluster_id": cluster_id,
                "cluster_color": cluster_color_map[cluster_id],
                "ancla_asociado": "" if pd.isna(associated_anchor) else str(associated_anchor).strip(),
            },
        })

geojson = {"type": "FeatureCollection", "features": features}
output_dir = Path.cwd()
if not (output_dir / "public").is_dir() and (output_dir / "mapa-AGEB-" / "public").is_dir():
    output_dir = output_dir / "mapa-AGEB-"
output_path = output_dir / "public" / "cluster_hbc_anclas.geojson"
output_path.parent.mkdir(parents=True, exist_ok=True)
with output_path.open("w", encoding="utf-8") as output_file:
    json.dump(geojson, output_file, ensure_ascii=False, indent=2)

print(f"GeoJSON exportado: {output_path} ({len(features)} puntos)")

GeoJSON exportado: c:\Users\jose.valdez\Downloads\nuevo-mapag\mapa-AGEB-\public\cluster_hbc_anclas.geojson (271 puntos)


In [12]:
from pathlib import Path

rutas_viales = gpd.read_file(ruta_nueva)
rutas_viales = rutas_viales.loc[rutas_viales.geometry.notna()].copy()
rutas_viales["cluster_id"] = rutas_viales["nombre_cluster"].astype(str).str.rsplit("_", n=1).str[-1]
rutas_viales = rutas_viales[["clues_imb_hbc", "clues_imb_ancla", "cluster_id", "geometry"]]

output_dir = Path.cwd()
if not (output_dir / "public").is_dir() and (output_dir / "mapa-AGEB-" / "public").is_dir():
    output_dir = output_dir / "mapa-AGEB-"
route_output_path = output_dir / "public" / "cluster_rutas.geojson"
route_output_path.parent.mkdir(parents=True, exist_ok=True)
rutas_viales.to_file(route_output_path, driver="GeoJSON")
print(f"Rutas viales exportadas: {route_output_path} ({len(rutas_viales)} rutas)")

Rutas viales exportadas: c:\Users\jose.valdez\Downloads\nuevo-mapag\mapa-AGEB-\public\cluster_rutas.geojson (186 rutas)
